In [1]:
from dataloader_helper import read_file, _pad
import numpy as np
import pandas as pd
import vector
import awkward as ak
from utils import get_clean_phi
import pyarrow as pa
import energyflow as ef

In [2]:
local_dataset_path = ''

# please create a local_dataset_path.txt file that contains the path to the folder that contains the JetClass Dataset
with open('local_dataset_path.txt') as f:
    local_dataset_path = f.read()

In [3]:
file_names = ['HToBB', 'HToCC', 'HToGG', 'HToWW4Q', 'TTBar', 'WToQQ', 'ZToQQ']

for file in file_names:
    x_particles, x_jets, y = read_file(local_dataset_path + file + '_120.root', 150, particle_features=['part_pt', 'part_eta', 'part_phi', 'part_mass'], jet_features=['jet_eta', 'jet_phi'])
    mini_dataset_x = np.transpose(x_particles[:1000], axes=(0,2,1))
    
    mini_dataset_jet_eta = x_jets[:1000, 0]
    
    mini_dataset_jet_phi = x_jets[:1000, 1]
    
    mask = mini_dataset_x[..., 3] < 0
    mini_dataset_x[mask, 3] = 0
    
    for i in range(mini_dataset_x.shape[0]):
        mask = mini_dataset_x[i][:, 0] > 0
        
        # before = np.copy(mini_dataset_x[i])
         
        mini_dataset_x[i, mask, 1] -= mini_dataset_jet_eta[i]
        mini_dataset_x[i, mask, 2] -= mini_dataset_jet_phi[i]
        mini_dataset_x[i, mask, 2] = get_clean_phi(mini_dataset_x[i, mask, 2], range='unsigned')
        
        mini_dataset_x[i] = ef.rotate_ptyphims(mini_dataset_x[i], rotate='ptscheme', copy=False)
        # after = np.copy(mini_dataset_x[i])
        
        # event_phi = mini_dataset_x[i, :, 2]
        # min_nonzero = event_phi[event_phi != 0].min()
        # max_nonzero = event_phi[event_phi != 0].max()
        
        # assert min_nonzero >= np.pi / 2, f'{file} {mini_dataset_jet_phi[i]}\n{before}\n{after}'
        # assert max_nonzero <= np.pi * 3 / 2, f'{file} {mini_dataset_jet_phi[i]}\n{before}\n{after}'
    
    mini_dataset_y = np.argmax(y[:1000], axis=1)
    
    # since we are skiping over the H->lvqq jet and the default order of the labels is such that H->lvqq is labeled as 5, we subtract 1 for each of the subsequent kinds of jets that comes after 5. 
    mini_dataset_y[mini_dataset_y > 5] -= 1
    
    np.savez(file+'_1k.npz', x=mini_dataset_x, y=mini_dataset_y)

In [4]:
qg_dataset_name = 'train_file_0.parquet'

df = pd.read_parquet('../dataset/'+qg_dataset_name)

In [5]:
table = pa.Table.from_pandas(
    df[["part_px", "part_py", "part_pz", "part_energy"]],
    preserve_index=False
)

jagged = ak.from_arrow(table)
p4 = vector.zip({
        "px": jagged["part_px"],
        "py": jagged["part_py"],
        "pz": jagged["part_pz"],
        "energy" : jagged["part_energy"],
    }
)

jet_eta = df['jet_eta'].values
jet_phi = df['jet_phi'].values

In [6]:
l = [p4.pt, p4.eta, p4.phi, p4.mass]

x_particles = np.stack([ak.to_numpy(_pad(a, maxlen=150)) for a in l], axis=1)

In [7]:
mini_dataset_x = np.transpose(x_particles[:1000], axes=(0,2,1))

mask = mini_dataset_x[..., 3] < 0
mini_dataset_x[mask, 3] = 0

mini_dataset_jet_eta = jet_eta[:1000]
mini_dataset_jet_phi = jet_phi[:1000]

for i in range(mini_dataset_x.shape[0]):
    mask = mini_dataset_x[i][:, 0] > 0
    
    mini_dataset_x[i, mask, 1] -= mini_dataset_jet_eta[i]
    mini_dataset_x[i, mask, 2] -= mini_dataset_jet_phi[i]
    mini_dataset_x[i, mask, 2] = get_clean_phi(mini_dataset_x[i, mask, 2], range='unsigned')
    
    mini_dataset_x[i] = ef.rotate_ptyphims(mini_dataset_x[i], rotate='ptscheme', copy=False)

mini_dataset_y = np.zeros(1000)

np.savez('QCD_1k.npz', x=mini_dataset_x, y=mini_dataset_y)

In [8]:
# saving the class labels to the npz file
# a dictionary of what the class labels represents to the npz file
class_labels = {
    'QCD'   : 0,
    'HToBB'  : 1, 
    'HToCC'  : 2, 
    'HToGG'  : 3, 
    'HToWW4Q': 4, 
    'TTBar'  : 5, 
    'WToQQ'  : 6, 
    'ZToQQ'  : 7
}

np.savez('class_labels.npz', **class_labels)